# 14.2 Python's Built-ins and Their Real Costs

**Prerequisites:** 14.1 Complexity Analysis, 02 Datatypes  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- The complexity of **every operation you use daily** - list, dict, set, tuple
- 🔴 Why `list.insert(0, x)` and `del data[0]` are O(n)
- 🔴 `in` on a list vs a set - the single most common accidental O(n²)
- **`collections.deque`** - O(1) at both ends
- How `dict` and `set` actually work, and when O(1) degrades
- 🔴 String concatenation in a loop, and `str.join`
- `bisect`, `heapq`, `Counter`, `defaultdict`, `OrderedDict`
- Choosing the right built-in - a decision table
- Interview questions on Python internals

---

## Why this comes before any data structure you write yourself

You will implement linked lists, trees and heaps in the notebooks that follow. In real work you will almost never use them — you will use `list`, `dict`, `set` and `deque`, because they are implemented in C and are extremely fast.

**What actually matters professionally is knowing their costs.** Almost every accidental O(n²) in real Python comes from one of four mistakes:

1. `x in some_list` inside a loop
2. `list.insert(0, x)` or `del list[0]` inside a loop
3. `result += piece` on a string inside a loop
4. `list.pop(0)` used as a queue

All four look innocent. None of them are.

> This notebook is the highest ratio of *practical value* to *effort* in the whole folder. The rest teaches you to build structures; this teaches you to use the ones you already have without accidentally making them quadratic.

## The reference table

### `list` — a dynamic array of pointers

| Operation | Complexity | Note |
|---|---|---|
| `data[i]` | **O(1)** | it is an array; index arithmetic |
| `data[i] = x` | **O(1)** | |
| `data.append(x)` | **O(1)** amortised | **14.1** |
| `data.pop()` | **O(1)** | from the **end** |
| 🔴 `data.insert(0, x)` | **O(n)** | shifts everything right |
| 🔴 `data.pop(0)` | **O(n)** | shifts everything left |
| 🔴 `del data[0]` | **O(n)** | same shift |
| 🔴 `x in data` | **O(n)** | scans |
| `data.sort()` | O(n log n) | Timsort (**14.10**) |
| `len(data)` | **O(1)** | stored, not counted |
| `data[a:b]` | O(b−a) | slicing **copies** |

### `dict` and `set` — hash tables

| Operation | Average | Worst | Note |
|---|---|---|---|
| `d[key]`, `key in d` | **O(1)** | O(n) | worst case needs pathological collisions |
| `d[key] = v`, `del d[key]` | **O(1)** | O(n) | |
| `len(d)` | **O(1)** | | |
| iterate | O(n) | | insertion order since **3.7** |

### `collections.deque` — a doubly linked list of blocks

| Operation | Complexity | vs list |
|---|---|---|
| `append` / `pop` | **O(1)** | same |
| **`appendleft` / `popleft`** | **O(1)** | 🔴 list is **O(n)** |
| `d[i]` | **O(n)** | 🔴 list is **O(1)** |

**That last row is the trade.** `deque` buys O(1) at both ends by giving up O(1) random access. Use it as a queue or stack; never as an array.

### 🔴 The one that costs most people the most time

```
    if item in my_list:      O(n)  - scans every element
    if item in my_set:       O(1)  - one hash lookup
    if key  in my_dict:      O(1)  - one hash lookup
```

Identical syntax. Completely different cost. Put the first one inside a loop over `n` items and you have written O(n²) that will pass every test you run on small data.

The cell below counts and times it.

In [ ]:
import time

N = 20_000
haystack_list = list(range(N))
haystack_set = set(haystack_list)
haystack_dict = {x: True for x in haystack_list}

LOOKUPS = 2_000
targets = [N - 1] * LOOKUPS          # worst case: the last element

started = time.perf_counter()
for t in targets:
    _ = t in haystack_list
list_time = time.perf_counter() - started

started = time.perf_counter()
for t in targets:
    _ = t in haystack_set
set_time = time.perf_counter() - started

started = time.perf_counter()
for t in targets:
    _ = t in haystack_dict
dict_time = time.perf_counter() - started

print(f"{LOOKUPS:,} lookups in a collection of {N:,}\n")
print(f"  list  {list_time * 1000:9.2f} ms")
print(f"  set   {set_time * 1000:9.2f} ms   {list_time / set_time:>8,.0f}x faster")
print(f"  dict  {dict_time * 1000:9.2f} ms   {list_time / dict_time:>8,.0f}x faster")
print()
print("The gap GROWS with N: the set stays O(1) while the list stays O(n).")
print("At N=20,000 it is already thousands of times. At N=1,000,000 the")
print("list version is unusable and the set version is unchanged.")

### The fix, in one line

```
    # 🔴 O(n * m)
    dupes = [x for x in a if x in b]           # b is a list

    # ✅ O(n + m)
    b_set = set(b)                             # O(m) once
    dupes = [x for x in a if x in b_set]       # O(1) each
```

Building the set costs O(m) **once**. After that every lookup is O(1). This one transformation turns more accidental O(n²) into O(n) than any other change you can make to Python code.

In [ ]:
def find_common_slow(a, b):
    return [x for x in a if x in b]            # b is a list -> O(n*m)


def find_common_fast(a, b):
    lookup = set(b)                            # O(m) once
    return [x for x in a if x in lookup]       # O(1) each -> O(n+m)


a = list(range(0, 6_000, 2))
b = list(range(0, 6_000, 3))

started = time.perf_counter()
slow_result = find_common_slow(a, b)
slow = time.perf_counter() - started

started = time.perf_counter()
fast_result = find_common_fast(a, b)
fast = time.perf_counter() - started

print("same answer      :", slow_result == fast_result, f"({len(slow_result)} items)")
print(f"list membership  : {slow * 1000:8.2f} ms")
print(f"set  membership  : {fast * 1000:8.2f} ms   {slow / fast:.0f}x faster")
print()
print("One extra line - set(b) - changed O(n*m) into O(n+m).")
print("\nEven better when you only need the overlap:")
print("   set(a) & set(b)  ->", len(set(a) & set(b)), "items, O(n+m)")

## 🔴 The front of a list is expensive

A `list` is a contiguous array. Removing or inserting at the **front** means every other element must move.

```
    data.pop(0)          [A][B][C][D]
                          ^  \  \  \      every element shifts left: O(n)
                             [B][C][D]

    deque.popleft()      the block just forgets its first slot: O(1)
```

This is why **using a list as a queue is a classic bug**. Appending is O(1), so it feels fine — but `pop(0)` makes the whole loop O(n²).

`collections.deque` is the fix, and it is a drop-in replacement for this use.

In [ ]:
from collections import deque

SIZE = 30_000   # 60k made this cell take 8.6 seconds; the point lands at 30k

# ---- list as a queue: O(n) per pop ----
queue_list = list(range(SIZE))
started = time.perf_counter()
while queue_list:
    queue_list.pop(0)                    # 🔴 shifts everything, every time
list_queue = time.perf_counter() - started

# ---- deque as a queue: O(1) per pop ----
queue_deque = deque(range(SIZE))
started = time.perf_counter()
while queue_deque:
    queue_deque.popleft()                # ✅ O(1)
deque_queue = time.perf_counter() - started

print(f"draining a queue of {SIZE:,}\n")
print(f"  list.pop(0)     {list_queue * 1000:9.1f} ms")
print(f"  deque.popleft() {deque_queue * 1000:9.1f} ms   "
      f"{list_queue / deque_queue:,.0f}x faster")
print()
print("list.pop(0) is O(n) -> the loop is O(n^2)")
print("deque.popleft() is O(1) -> the loop is O(n)")
print()
print("🔴 And the trade: deque gives up O(1) indexing.")

sample_list = list(range(100_000))
sample_deque = deque(sample_list)
middle = 50_000

started = time.perf_counter()
for _ in range(2_000):
    _ = sample_list[middle]
list_index = time.perf_counter() - started

started = time.perf_counter()
for _ in range(2_000):
    _ = sample_deque[middle]
deque_index = time.perf_counter() - started

print(f"  list[50_000]    {list_index * 1000:9.2f} ms   O(1)")
print(f"  deque[50_000]   {deque_index * 1000:9.2f} ms   O(n) - "
      f"{deque_index / list_index:.0f}x slower")
print("\n  Use deque for queues and stacks. Use list for indexing.")

## 🔴 Building a string in a loop

Strings are **immutable**. `result += piece` cannot extend the existing string — it allocates a **new** one and copies both parts in.

```
    result += piece      allocate len(result)+len(piece), copy both: O(len(result))
```

Do that `n` times and the total is 1 + 2 + 3 + … + n = **O(n²)**.

`str.join` walks the pieces once to compute the total length, allocates **once**, and copies each piece exactly once: **O(total)**.

> ⚠️ You may not always see the quadratic behaviour. CPython has an optimisation that can extend a string in place when the old one has no other references — but it is an implementation detail that silently stops applying (a second reference, a different build, PyPy). Never rely on it; the fix costs nothing.

In [ ]:
PIECES = ["line " + str(i) for i in range(40_000)]
# ---- concatenation in a loop ----
started = time.perf_counter()
built = ""
for piece in PIECES:
    built += piece
concat_time = time.perf_counter() - started
# ---- str.join ----
started = time.perf_counter()
joined = "".join(PIECES)
join_time = time.perf_counter() - started
print("identical result:", built == joined, f"({len(joined):,} characters)")
print(f"  += in a loop  {concat_time * 1000:8.2f} ms")
print(f"  str.join      {join_time * 1000:8.2f} ms   "
      f"{concat_time / join_time:,.0f}x faster")
print()
# The in-place optimisation, and how easily it is defeated
started = time.perf_counter()
built = ""
keep_a_reference = []
for piece in PIECES[:8_000]:
    built += piece
    keep_a_reference.append(built)       # a second reference each time
defeated = time.perf_counter() - started
started = time.perf_counter()
built2 = ""
for piece in PIECES[:8_000]:
    built2 += piece
plain = time.perf_counter() - started
print("same loop, 8,000 pieces:")
print(f"  += alone                {plain * 1000:8.2f} ms")
print(f"  += with a live reference{defeated * 1000:8.2f} ms   "
      f"{defeated / plain:,.0f}x slower")
print("\n  ^ the in-place optimisation vanished the moment the old string")
print("    had another reference. Use str.join and stop thinking about it.")

## How `dict` and `set` get O(1)

Both are **hash tables**. Storing `d[key] = value`:

```
   1. h = hash(key)             turn the key into an integer
   2. slot = h % table_size     pick a bucket
   3. if occupied by a DIFFERENT key -> probe for the next free slot
   4. store (key, value)
```

Lookup repeats steps 1–3. With few collisions that is a constant number of steps, regardless of size — hence O(1).

### What this requires of your keys

| Requirement | Consequence |
|---|---|
| **Hashable** | immutable in practice: `str`, `int`, `tuple`, `frozenset` |
| `a == b` ⟹ `hash(a) == hash(b)` | if you write `__eq__`, write `__hash__` (**5.1**) |
| hash must not change | 🔴 mutating a key after insertion loses the entry |

### When O(1) degrades

If many keys collide, lookups walk a chain and approach O(n). With well-behaved keys this does not happen by accident — it happens **adversarially**, which is why Python randomises string hashing per process by default (`PYTHONHASHSEED`) to prevent hash-collision denial-of-service attacks.

In [ ]:
# ---- what is hashable ----
candidates = [42, "text", (1, 2), frozenset([1]), [1, 2], {1: 2}, {1, 2}]
for obj in candidates:
    try:
        hash(obj)
        print(f"  {type(obj).__name__:<10} hashable")
    except TypeError:
        print(f"  {type(obj).__name__:<10} NOT hashable - it is mutable")

# ---- 🔴 mutating a key destroys the lookup ----
print("\nwhat happens if a key's hash changes:")


class Sloppy:
    """Hashes on a field that can be changed - a real bug pattern."""

    def __init__(self, name):
        self.name = name

    def __hash__(self):
        return hash(self.name)

    def __eq__(self, other):
        return isinstance(other, Sloppy) and self.name == other.name


key = Sloppy("cache-a")
registry = {key: "payload"}
print("  before mutation, found:", key in registry)

key.name = "cache-b"                    # the hash just changed
print("  after  mutation, found:", key in registry)
print("  but it is still stored :", len(registry) == 1)
print("  ^ the entry is in a bucket the new hash never looks in.")
print("    It is unreachable and un-deletable. Keys must be immutable.")

## The rest of the toolkit

| Tool | For | Complexity |
|---|---|---|
| `collections.Counter` | counting occurrences | O(n) to build |
| `collections.defaultdict` | grouping without `if key not in d` | same as dict |
| `collections.deque` | queue / stack / sliding window | O(1) both ends |
| `heapq` | smallest-first, top-k (**14.8**) | O(log n) push/pop |
| `bisect` | insert into / search a **sorted** list | O(log n) search, O(n) insert |
| `frozenset` | a set usable as a dict key | as set |

🔴 **`bisect.insort` is O(n), not O(log n).** Finding the position is O(log n); actually inserting still shifts the array. If you are inserting constantly, you want a heap (**14.8**) or a balanced tree (**14.7**), not a sorted list.

In [ ]:
import bisect
from collections import Counter, defaultdict

words = "deploy build deploy test build deploy release test deploy".split()

# ---- Counter ----
counts = Counter(words)
print("Counter          :", counts)
print("  most_common(2) :", counts.most_common(2))

# ---- the manual version this replaces ----
manual = {}
for word in words:
    manual[word] = manual.get(word, 0) + 1
print("  same as manual :", manual == dict(counts))

# ---- defaultdict for grouping ----
jobs = [("search", "reindex"), ("platform", "purge"),
        ("search", "warm"), ("growth", "digest")]
by_team = defaultdict(list)
for team, job in jobs:
    by_team[team].append(job)            # no 'if team not in by_team'
print("\ndefaultdict     :", dict(by_team))

# ---- bisect on sorted data ----
sorted_scores = [10, 20, 30, 40, 50]
print("\nbisect on", sorted_scores)
print("  position for 35 :", bisect.bisect_left(sorted_scores, 35), "  O(log n)")
print("  is 30 present?  :", sorted_scores[bisect.bisect_left(sorted_scores, 30)] == 30)

bisect.insort(sorted_scores, 35)
print("  after insort    :", sorted_scores)
print("  🔴 insort is O(n) - the SEARCH is O(log n), the shift is not.")

## Choosing

| You need | Use | Why |
|---|---|---|
| Ordered, indexed, mostly appending | `list` | O(1) index and append |
| Membership tests | `set` | O(1) instead of O(n) |
| Key → value | `dict` | O(1), insertion-ordered since 3.7 |
| A queue (FIFO) | `deque` | O(1) `popleft`; list is O(n) |
| A stack (LIFO) | `list` | `append`/`pop` are both O(1) |
| Counting | `Counter` | one line, O(n) |
| Grouping | `defaultdict(list)` | no key-existence checks |
| Smallest/largest repeatedly | `heapq` | O(log n) (**14.8**) |
| A fixed record | `tuple` / `NamedTuple` | immutable, hashable, smaller |
| An immutable set | `frozenset` | usable as a dict key |

### The rule that catches most of it

> **If you are searching inside a loop, you are probably writing O(n²).** Build a `set` or `dict` before the loop instead.

## Interview questions

**1. What is the time complexity of `list.append` and why?**
> Amortised O(1). The list over-allocates, so reallocations become geometrically rarer (**14.1**).

**2. Why is `list.pop(0)` O(n) but `list.pop()` O(1)?**
> A list is a contiguous array. Removing from the front shifts every remaining element left; removing from the end shifts nothing.

**3. How does a Python `dict` achieve O(1) lookup?**
> Hash table: hash the key to pick a bucket, probe on collision. Constant steps when collisions are rare. Worst case is O(n) with pathological collisions.

**4. What makes an object usable as a dict key?**
> It must be hashable and its hash must not change. Equal objects must have equal hashes. In practice that means immutable.

**5. `list` vs `deque` — when would you choose each?**
> `deque` for O(1) operations at both ends — queues, sliding windows. `list` for O(1) indexing. `deque` indexing is O(n).

**6. Why is building a string with `+=` in a loop bad?**
> Strings are immutable, so each `+=` allocates and copies — O(n²) overall. `str.join` allocates once: O(n).

**7. Given two lists, find the common elements efficiently.**
> `set(a) & set(b)` — O(n+m). The naive nested membership test is O(n·m). This is the single most common optimisation in real Python.

**8. Is `dict` ordered?**
> Insertion-ordered since 3.7 (an implementation detail in 3.6). `OrderedDict` still exists for `move_to_end` and order-sensitive equality.

**9. What is the complexity of slicing?**
> O(k) for a slice of length k, because slicing **copies**. Slicing inside a loop is a quiet way to add a factor of n.

**10. How would you find the 10 largest items in a list of a million?**
> `heapq.nlargest(10, data)` — O(n log 10), not O(n log n). Sorting the whole list to take 10 is wasteful (**14.8**).

---

## Common Mistakes & Pitfalls

1. 🔴 **`x in some_list` inside a loop.** O(n) each, O(n²) total. Build a `set` first.
2. 🔴 **`list.pop(0)` or `insert(0, x)` as a queue.** Both O(n). Use `deque`.
3. 🔴 **`result += piece` on a string in a loop.** O(n²). Use `str.join`.
4. 🔴 **Mutating an object used as a dict key.** Its hash changes and the entry becomes unreachable.
5. **Assuming `bisect.insort` is O(log n).** The search is; the insertion still shifts the array, so it is O(n).
6. **Using `deque` for indexed access.** `deque[i]` is O(n).
7. **Sorting to get the top k.** O(n log n) where `heapq.nlargest` is O(n log k).
8. **Slicing inside a loop.** Each slice copies - a hidden factor of n.
9. **Defining `__eq__` without `__hash__`.** The class becomes unhashable, or worse, inconsistent.

## Best Practices

- Know the complexity of the operations you use most; it is a short list.
- Convert to a `set` before repeated membership tests.
- Use `deque` for queues, `list` for stacks and indexing.
- Build strings with `str.join`, or `io.StringIO` for incremental writing.
- Use `Counter` and `defaultdict` instead of hand-rolled dict bookkeeping.
- Use `heapq.nlargest`/`nsmallest` for top-k rather than sorting everything.
- Keep dict and set keys immutable.
- Prefer built-in C-implemented operations over hand-written Python loops (**14.1**).
- When a loop feels slow, look for a hidden O(n) operation inside it before anything else.

## Practice Exercises

Try these before moving on.

1. Time `x in list` versus `x in set` at N = 1,000 / 10,000 / 100,000. Does the *ratio* grow with N? Explain why using **14.1**.
2. 🔴 Write a function that removes duplicates from a list while preserving order, twice: once with a list for 'seen' and once with a set. Compare by doubling n.
3. Implement a fixed-size sliding window over a stream using `deque(maxlen=k)`. What does `maxlen` do when the window is full?
4. Build a 100,000-line string with `+=` and with `str.join`. Then repeat the `+=` version while keeping a reference to each intermediate string. Explain the difference.
5. Write a class with `__eq__` but no `__hash__` and try to put it in a set. What happens, and why is that the safe default?
6. Use `Counter` to find the 5 most common words in a large text, then write the same thing with a plain dict. Which is clearer, and are they the same complexity?
7. 🔴 You have a list of a million records and need to look items up by id thousands of times. Describe the change you would make and the complexity before and after.
8. Measure `sorted(data)[-10:]` against `heapq.nlargest(10, data)` for a million items. Explain the gap in terms of O(n log n) versus O(n log k).